# Notebook 23 — Negativos difíciles de LVIS para Modelo C

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

## Objetivo

Extraer de LVIS v1 imágenes con máscaras de segmentación de los tres confundidores principales del pipeline:

| Categoría LVIS | Categorías GAR problemáticas | FP% en Modelo B |
|----------------|-----------------------------|-----------------|
| `cellular_telephone` | N6, N7, N8, N9 (phone) | 50–78% |
| `bottle` / `water_bottle` / `beer_bottle` | N10, N11 (bottle/drinking) | 22–44% |
| `remote_control` | N8, N9 (phone recording) | similar morfología |

Estas imágenes se añadirán al dataset de entrenamiento de Modelo B con clase `no_weapon` para entrenar **Modelo C**.

## Flujo

```
LVIS v1 annotations JSON
  → filtrar categorías objetivo
  → filtrar imágenes que también contengan persona (contexto realista)
  → descargar imágenes desde COCO URLs
  → convertir máscaras LVIS (polígonos) → formato YOLOv8 segmentación
  → guardar en estructura de dataset YOLOv8
```

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install lvis ultralytics opencv-python matplotlib
print('✅ Dependencias instaladas')

In [ ]:
import json
import os
import shutil
import urllib.request
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── CONFIG ────────────────────────────────────────────────────────────────────
OUT_DIR      = '/content/drive/MyDrive/TFM/datasets/lvis_negatives'
LVIS_ANN_DIR = '/content/lvis_annotations'

# Categorías objetivo en LVIS
TARGET_CATEGORIES = [
    'cellular_telephone',
    'bottle',
    'water_bottle',
    'beer_bottle',
    'remote_control',
]

# Máximo de imágenes por categoría (para no sobrecargar)
MAX_IMAGES_PER_CAT = 300

# Solo imágenes que también contengan persona (contexto realista de vigilancia)
REQUIRE_PERSON = True

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(LVIS_ANN_DIR).mkdir(parents=True, exist_ok=True)

print('✅ Config cargada')
print(f'   Categorías objetivo: {TARGET_CATEGORIES}')
print(f'   Max imágenes/cat:    {MAX_IMAGES_PER_CAT}')
print(f'   Requiere persona:    {REQUIRE_PERSON}')

---
## 1. Descargar anotaciones LVIS v1

In [ ]:
# Descargar anotaciones train y val de LVIS v1
LVIS_URLS = {
    'train': 'https://dl.fbaipublicfiles.com/LVIS/lvis_v1_train.json.zip',
    'val':   'https://dl.fbaipublicfiles.com/LVIS/lvis_v1_val.json.zip',
}

for split, url in LVIS_URLS.items():
    zip_path = f'{LVIS_ANN_DIR}/lvis_v1_{split}.json.zip'
    json_path = f'{LVIS_ANN_DIR}/lvis_v1_{split}.json'

    if Path(json_path).exists():
        print(f'  ✅ {split}: ya existe')
        continue

    print(f'  Descargando {split}...', end=' ', flush=True)
    urllib.request.urlretrieve(url, zip_path)
    print('descomprimiendo...', end=' ', flush=True)
    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(LVIS_ANN_DIR)
    os.remove(zip_path)
    print('✅')

print('✅ Anotaciones LVIS descargadas')

---
## 2. Explorar categorías disponibles

In [ ]:
# Cargar anotaciones train (el split principal)
print('Cargando anotaciones train...')
with open(f'{LVIS_ANN_DIR}/lvis_v1_train.json') as f:
    lvis_train = json.load(f)
print(f'  Imágenes train: {len(lvis_train["images"])}')
print(f'  Anotaciones:    {len(lvis_train["annotations"])}')
print(f'  Categorías:     {len(lvis_train["categories"])}')

# Buscar categorías objetivo
cat_map = {cat['name']: cat for cat in lvis_train['categories']}
print()
print('Categorías objetivo encontradas en LVIS:')
print(f'  {"Nombre":<30} {"ID":>6} {"Frecuencia":>12} {"Instancias":>12}')
print('  ' + '-'*65)

found_cats = {}
for target in TARGET_CATEGORIES:
    if target in cat_map:
        cat = cat_map[target]
        found_cats[target] = cat
        print(f'  {target:<30} {cat["id"]:>6} {cat.get("frequency","?"):>12} {cat.get("instance_count","?"):>12}')
    else:
        # Buscar similares
        similar = [c['name'] for c in lvis_train['categories']
                   if target.split('_')[0] in c['name']]
        print(f'  {target:<30}  ❌ No encontrado. Similares: {similar[:5]}')

# Buscar también 'person' para el filtro
person_cats = [c for c in lvis_train['categories']
               if 'person' in c['name'].lower() or c['name'] == 'person']
print(f'\nCategorías de persona disponibles: {[c["name"] for c in person_cats[:5]]}')

---
## 3. Filtrar imágenes con los objetos objetivo + persona

In [ ]:
# IDs de categorías objetivo
target_cat_ids = {cat['id'] for cat in found_cats.values()}

# ID de persona
person_cat_ids = {c['id'] for c in lvis_train['categories']
                  if c['name'] in ['person', 'man', 'woman', 'boy', 'girl']}
print(f'IDs de persona: {person_cat_ids}')
print(f'IDs objetivo:   {target_cat_ids}')

# Índice: image_id → lista de category_ids
img_to_cats = defaultdict(set)
img_to_anns = defaultdict(list)
for ann in lvis_train['annotations']:
    img_to_cats[ann['image_id']].add(ann['category_id'])
    img_to_anns[ann['image_id']].append(ann)

# Filtrar imágenes
selected = defaultdict(list)  # cat_name → [image_ids]
for img_id, cat_ids in img_to_cats.items():
    has_target  = cat_ids & target_cat_ids
    has_person  = cat_ids & person_cat_ids
    if has_target and (not REQUIRE_PERSON or has_person):
        for target_cat_id in has_target:
            cat_name = next(c['name'] for c in found_cats.values()
                            if c['id'] == target_cat_id)
            selected[cat_name].append(img_id)

print()
print('Imágenes seleccionadas por categoría:')
total = 0
for cat_name, img_ids in sorted(selected.items()):
    n = min(len(img_ids), MAX_IMAGES_PER_CAT)
    total += n
    print(f'  {cat_name:<30}: {len(img_ids):>5} disponibles → {n} seleccionadas')
print(f'  Total: {total} imágenes')

---
## 4. Descargar imágenes y preparar dataset

In [ ]:
# Índice image_id → coco_url
img_id_to_info = {img['id']: img for img in lvis_train['images']}

# Estructura de carpetas YOLOv8 segmentación
# lvis_negatives/
#   images/train/
#   labels/train/   ← formato YOLOv8 seg: class x1 y1 x2 y2 ... (polígono normalizado)

IMG_DIR = Path(OUT_DIR) / 'images' / 'train'
LBL_DIR = Path(OUT_DIR) / 'labels' / 'train'
IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)

# Clase 0 = no_weapon (negativo difícil)
CLASS_ID = 0

def polygon_to_yolo_seg(segmentation, img_w, img_h):
    """
    Convierte polígono LVIS/COCO [x1,y1,x2,y2,...] a formato YOLOv8 segmentación.
    Normaliza coordenadas a [0,1].
    Devuelve string: 'class x1_norm y1_norm x2_norm y2_norm ...'
    """
    coords = []
    for i in range(0, len(segmentation), 2):
        x = segmentation[i] / img_w
        y = segmentation[i+1] / img_h
        x = max(0.0, min(1.0, x))
        y = max(0.0, min(1.0, y))
        coords.extend([x, y])
    return ' '.join([str(CLASS_ID)] + [f'{v:.6f}' for v in coords])


downloaded = 0
errors     = 0
skipped    = 0

# Recopilar todos los image_ids seleccionados (con límite por categoría)
all_selected_ids = set()
for cat_name, img_ids in selected.items():
    # Shuffle para variedad
    import random
    random.seed(42)
    sample = random.sample(img_ids, min(len(img_ids), MAX_IMAGES_PER_CAT))
    all_selected_ids.update(sample)

print(f'Total imágenes únicas a descargar: {len(all_selected_ids)}')
print('Descargando...')

for i, img_id in enumerate(sorted(all_selected_ids)):
    img_info = img_id_to_info[img_id]
    url      = img_info['coco_url']
    img_w    = img_info['width']
    img_h    = img_info['height']
    fname    = f'lvis_{img_id:012d}.jpg'
    img_path = IMG_DIR / fname
    lbl_path = LBL_DIR / fname.replace('.jpg', '.txt')

    # Skip si ya existe
    if img_path.exists():
        skipped += 1
        continue

    try:
        urllib.request.urlretrieve(url, img_path)
    except Exception as e:
        errors += 1
        continue

    # Generar label — solo anotaciones de categorías objetivo en esta imagen
    anns = img_to_anns[img_id]
    lines = []
    for ann in anns:
        if ann['category_id'] not in target_cat_ids:
            continue
        if not ann.get('segmentation'):
            continue
        # LVIS usa listas de polígonos (puede haber varios por instancia)
        for poly in ann['segmentation']:
            if len(poly) >= 6:  # mínimo 3 puntos
                line = polygon_to_yolo_seg(poly, img_w, img_h)
                lines.append(line)

    if lines:
        with open(lbl_path, 'w') as f:
            f.write('\n'.join(lines))
        downloaded += 1
    else:
        img_path.unlink()  # borrar imagen sin anotaciones válidas

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(all_selected_ids)} — descargadas:{downloaded} errores:{errors} skip:{skipped}')

print(f'\n✅ Descarga completada')
print(f'   Imágenes descargadas: {downloaded}')
print(f'   Errores:              {errors}')
print(f'   Ya existían:          {skipped}')

---
## 5. Verificación visual — muestra de imágenes con máscaras

In [ ]:
# Mostrar 12 imágenes con sus máscaras de segmentación
img_files = sorted(IMG_DIR.glob('*.jpg'))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
axes = axes.flatten()

COLORS = [(255,80,80), (80,255,80), (80,80,255),
          (255,255,80), (255,80,255), (80,255,255)]

for ax, img_path in zip(axes, img_files):
    lbl_path = LBL_DIR / img_path.name.replace('.jpg', '.txt')
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    overlay = img.copy()

    if lbl_path.exists():
        for li, line in enumerate(lbl_path.read_text().strip().split('\n')):
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            coords = list(map(float, parts[1:]))
            pts = np.array([[coords[i]*w, coords[i+1]*h]
                            for i in range(0, len(coords), 2)], dtype=np.int32)
            color = COLORS[li % len(COLORS)]
            cv2.fillPoly(overlay, [pts], color)
        img = cv2.addWeighted(img, 0.5, overlay, 0.5, 0)

    ax.imshow(img)
    ax.set_title(img_path.name[5:15], fontsize=7)
    ax.axis('off')

plt.suptitle('Muestra de negativos difíciles LVIS con máscaras de segmentación', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/muestra_negativos_lvis.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Muestra guardada')

---
## 6. Estadísticas del dataset generado

In [ ]:
img_files = list(IMG_DIR.glob('*.jpg'))
lbl_files = list(LBL_DIR.glob('*.txt'))
total_masks = sum(len(f.read_text().strip().split('\n')) for f in lbl_files if f.stat().st_size > 0)

print('='*50)
print('ESTADÍSTICAS DEL DATASET LVIS NEGATIVOS')
print('='*50)
print(f'  Imágenes:          {len(img_files)}')
print(f'  Labels:            {len(lbl_files)}')
print(f'  Máscaras totales:  {total_masks}')
print(f'  Media máscaras/img: {total_masks/len(lbl_files):.1f}')
print()
print('  Directorio: ' + str(OUT_DIR))
print()
print('  Próximo paso: combinar con dataset de armas y entrenar Modelo C')

# Guardar YAML para YOLOv8
yaml_content = f"""# Dataset negativos difíciles LVIS
# Categorías: cellular_telephone, bottle, water_bottle, beer_bottle, remote_control
# Generado para TFM — Oliver Legarreta García

path: {OUT_DIR}
train: images/train
val: images/train  # se dividirá en el notebook de entrenamiento

nc: 1
names:
  0: no_weapon
"""
yaml_path = Path(OUT_DIR) / 'lvis_negatives.yaml'
yaml_path.write_text(yaml_content)
print(f'✅ YAML guardado en: {yaml_path}')

---
## Notas para el Notebook 24 (entrenamiento Modelo C)

El dataset generado aquí se combinará con el dataset de armas existente para entrenar **Modelo C**.

**Estrategia de combinación:**
- Imágenes positivas (armas): dataset original de Modelo B
- Negativos difíciles originales: ~3000 imágenes COCO (usadas en Modelo B)
- **Nuevos negativos difíciles LVIS**: imágenes generadas en este notebook

**Hipótesis:** al añadir máscaras de segmentación de los confundidores principales, el modelo aprenderá a discriminar por forma exacta del objeto, no solo por bounding box. Esto debería reducir los FP en las categorías N6–N9 (teléfono) y N10–N11 (botella).

**Evaluación:** mismo protocolo que Modelo B — 258 clips GAR, mismas métricas, misma tabla comparativa.